# 第4回実習：日本語AI推論＋UR5e＋Robotiq 2F-85の連続制御（MuJoCo）

**開発途中の技術実証・物体なし。** 既存の `01`、`02`、`03` ノートブックは変更しない。新しい04ノートブックのみを実行する。

- **AI部分**：Sentence Transformersの学習済み文章埋め込みモデルで日本語指示を類似度判定し、`LEFT_SAMPLE`／`RIGHT_SAMPLE`／`HOME`／`REJECT` を選ぶ（02を踏襲）。
- **ロボット部分**：MuJoCo Menagerieの公式UR5eとRobotiq 2F-85モデルを `MjSpec.attach()` で統合（03で成功した方法を踏襲）。
- **連続動作**：指示ごとに「UR5eが事前設定の姿勢へ移動 → グリッパーCLOSE → OPEN → HOME」。曖昧な指示（`REJECT`）には**新しい目標動作を指令しない**。
- **成果物**：2画面MP4、フレーム別CSV、指示別CSV。**把持・搬送・逆運動学・軌道計画・ROS 2は未実装。**

**注意**：AIはロボット関節軌道を生成しない。AIはタスク選択のみを行い、実際の制御は事前定義した関節目標に基づく。これは実機用の安全制御ではない。モデルやライブラリ更新で失敗する場合、まず02・03が動作するか切り分ける。

### 実行手順
Google Colabで「ランタイム → すべてのセルを実行」とする前に、まず**セル1～4**を順番に実行してモデル・静止画像まで確認する。その後セル5の短い連続動作テストへ進む。

## セル1：Colab CPU環境の準備
`MUJOCO_GL=osmesa` はMuJoCoのimportより先に指定する。GPUランタイムは不要。モデルの初回取得にはインターネット接続が必要。

In [ ]:
# セル1：依存ライブラリのインストール（Colab専用）
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y libosmesa6 libosmesa6-dev fonts-noto-cjk > /dev/null
%pip -q install "mujoco>=3.3,<4" "mujoco-menagerie==2026.10.3" "sentence-transformers>=3.0" "imageio[ffmpeg]" pillow pandas numpy matplotlib
print("セル1完了：依存ライブラリのインストール")

## セル2：日本語AI推論とREJECTの確認
02の設定を再使用する。登録例文に酷似した言い換えでも棄却が起こり得るので、事前の動作検証が必要。ここでは**あらかじめ選んだ4つの実演文**を用いる。実演の成功率を汎化性能として扱ってはならない。

In [ ]:
# セル2A: 02で動作した学習済みAIモデルを準備
import os
os.environ["MUJOCO_GL"] = "osmesa"  # mujoco を import するより前に設定
import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
text_model = SentenceTransformer(MODEL_ID, device="cpu")

# 作業IDと学習済みモデルへ比較させる日本語説明
TASK_TEXTS = {
    "LEFT_SAMPLE": "左の試料を測定場所へ移動して",
    "RIGHT_SAMPLE": "右の試料を測定場所へ移動して",
    "HOME": "アームを待機位置に戻して",
}
TASK_NAMES = list(TASK_TEXTS)
reference_embeddings = text_model.encode(
    [TASK_TEXTS[k] for k in TASK_NAMES], normalize_embeddings=True
)
print("AIモデル読み込み完了:", MODEL_ID)
print("登録タスク:", TASK_NAMES)

In [ ]:
# セル2B: 02で動作した推論・棄却判定
MIN_SCORE = 0.60
MIN_MARGIN = 0.04

def infer_task(instruction):
    start = time.perf_counter()
    vector = text_model.encode([instruction], normalize_embeddings=True)[0]
    values = reference_embeddings @ vector
    order = np.argsort(values)[::-1]
    best, second = int(order[0]), int(order[1])
    chosen = TASK_NAMES[best]
    score, margin = float(values[best]), float(values[best] - values[second])
    # 語の矛盾だけを検知する確認用チェック（この規則自体はAIではない）
    contradictory = ("右" in instruction and "左" in instruction)
    if chosen == "LEFT_SAMPLE" and ("右" in instruction or "試料B" in instruction):
        contradictory = True
    if chosen == "RIGHT_SAMPLE" and ("左" in instruction or "試料A" in instruction):
        contradictory = True
    accepted = score >= MIN_SCORE and margin >= MIN_MARGIN and not contradictory
    return {
        "instruction": instruction,
        "proposed_task": chosen,
        "task_id": chosen if accepted else "REJECT",
        "score": round(score, 4),
        "margin": round(margin, 4),
        "inference_seconds": round(time.perf_counter() - start, 4),
        "all_scores": {TASK_NAMES[i]: round(float(values[i]), 4) for i in range(len(TASK_NAMES))},
    }

TEST_INPUTS = [
    "左の試料を測定場所へ移動して",
    "右の試料を測定場所へ移動して",
    "アームを待機位置に戻して",
    "右と左のどちらかの試料を運んで",   # 曖昧な指示を棄却できるか確認
]
for s in TEST_INPUTS:
    d = infer_task(s)
    print(f"入力: {s} → {d['task_id']} | score={d['score']:.4f} margin={d['margin']:.4f}")
assert infer_task(TEST_INPUTS[0])["task_id"] == "LEFT_SAMPLE", "左の指示が棄却・誤判定されました。結果を確認してください。"
assert infer_task(TEST_INPUTS[1])["task_id"] == "RIGHT_SAMPLE", "右の指示が棄却・誤判定されました。結果を確認してください。"
assert infer_task(TEST_INPUTS[2])["task_id"] == "HOME", "待機指示が棄却・誤判定されました。結果を確認してください。"
print("明確な3指示の動作確認に成功しました。")

## セル3：UR5e＋2F-85のモデル接続
03で開閉が成功した構成をそのまま再利用する。アーム6軸とグリッパー1系統の計7アクチュエータが登録されたことを確認する。

In [ ]:
# セル3：第3回の実績があるモデル接続と初期姿勢の設定
import os
os.environ["MUJOCO_GL"] = "osmesa"   # mujocoをimportする前に指定

import mujoco
import mujoco_menagerie as mm
import numpy as np
import pandas as pd
from pathlib import Path

print("MuJoCo version:", mujoco.__version__)

# 公式MJCFを「編集可能なMjSpec」として取得
ur5e_spec = mm.get("universal_robots_ur5e").spec("scene")
gripper_spec = mm.get("robotiq_2f85").spec("2f85")

# UR5e公式モデルにはattachment_siteが定義されている
mount_site = ur5e_spec.site("attachment_site")
assert mount_site is not None, "UR5eのattachment_siteが見つかりません。"

# 接続先：UR5e wrist_3_linkのattachment_site
# prefixにより他のリンク名・材料名・アクチュエータ名との重複を防ぐ
ur5e_spec.attach(gripper_spec, prefix="grip_", site=mount_site)

# 結合モデルをコンパイル
model = ur5e_spec.compile()
data = mujoco.MjData(model)

# 結合モデルは「アーム6関節 + グリッパー8関節」、制御は「アーム6 + グリッパー1」の想定
assert model.nu == 7, f"制御アクチュエータの数が想定と異なります: nu={model.nu}"
assert model.nq >= 14, f"関節自由度が想定と異なります: nq={model.nq}"

ARM_JOINTS = [
    "shoulder_pan_joint", "shoulder_lift_joint", "elbow_joint",
    "wrist_1_joint", "wrist_2_joint", "wrist_3_joint",
]
ARM_ACTUATORS = [
    "shoulder_pan", "shoulder_lift", "elbow",
    "wrist_1", "wrist_2", "wrist_3",
]
HOME_Q = np.array([-1.5708, -1.5708, 1.5708, -1.5708, -1.5708, 0.0])

# 文字列名からMuJoCo内部の番号を取得。番号の並び順を暗黙に仮定しない。
def lookup(obj_type, name):
    number = mujoco.mj_name2id(model, obj_type, name)
    if number < 0:
        raise ValueError(f"モデル内に {name!r} がありません。アクチュエータ名を確認してください。")
    return int(number)

arm_joint_ids = [lookup(mujoco.mjtObj.mjOBJ_JOINT, n) for n in ARM_JOINTS]
arm_qpos_ids = [int(model.jnt_qposadr[i]) for i in arm_joint_ids]
arm_actuator_ids = [lookup(mujoco.mjtObj.mjOBJ_ACTUATOR, n) for n in ARM_ACTUATORS]

finger_actuator_id = lookup(mujoco.mjtObj.mjOBJ_ACTUATOR, "grip_fingers_actuator")
left_joint_id = lookup(mujoco.mjtObj.mjOBJ_JOINT, "grip_left_driver_joint")
right_joint_id = lookup(mujoco.mjtObj.mjOBJ_JOINT, "grip_right_driver_joint")
left_qpos_id = int(model.jnt_qposadr[left_joint_id])
right_qpos_id = int(model.jnt_qposadr[right_joint_id])
pinch_site_id = lookup(mujoco.mjtObj.mjOBJ_SITE, "grip_pinch")

# UR5eをHOME姿勢に設定。グリッパーは全開(制御入力0)から開始。
mujoco.mj_resetData(model, data)
for qid, target in zip(arm_qpos_ids, HOME_Q):
    data.qpos[qid] = target
for aid, target in zip(arm_actuator_ids, HOME_Q):
    data.ctrl[aid] = target
data.ctrl[finger_actuator_id] = 0.0
mujoco.mj_forward(model, data)

print(f"モデル統合成功: nq={model.nq}, nv={model.nv}, nu={model.nu}")
print("グリッパー制御アクチュエータ:", mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, finger_actuator_id))
print("制御範囲:", model.actuator_ctrlrange[finger_actuator_id].tolist())
print("グリッパーの指関節位置(初期):", float(data.qpos[left_qpos_id]), float(data.qpos[right_qpos_id]))
print("先端把持点の位置[m]:", np.round(data.site_xpos[pinch_site_id], 4))
print("★ 次はセル3の静止画像を確認してください。")

## セル4：装着モデルを静止画像で確認
アーム全体とグリッパー先端の2画面を表示する。ここでグリッパーが先端に接続されていることを確認する。

In [ ]:
# セル4：UR5e全体 / グリッパー拡大の2画面
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video
import imageio.v2 as imageio

W, H = 480, 360
renderer = mujoco.Renderer(model, height=H, width=W)

whole_camera = mujoco.MjvCamera()
whole_camera.type = mujoco.mjtCamera.mjCAMERA_FREE
whole_camera.azimuth = 135
whole_camera.elevation = -24
whole_camera.distance = 1.70
whole_camera.lookat[:] = [0, 0, 0.35]

close_camera = mujoco.MjvCamera()
close_camera.type = mujoco.mjtCamera.mjCAMERA_FREE
close_camera.azimuth = 135
close_camera.elevation = -15
close_camera.distance = 0.34

font_file = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font = ImageFont.truetype(font_file, 18) if Path(font_file).exists() else ImageFont.load_default()
small_font = ImageFont.truetype(font_file, 15) if Path(font_file).exists() else ImageFont.load_default()


def render_pair(phase="OPEN", cmd=0):
    """実機ではなく、MuJoCoの現在状態を2視点で撮影する。"""
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera=whole_camera)
    view1 = renderer.render().copy()
    close_camera.lookat[:] = data.site_xpos[pinch_site_id]
    renderer.update_scene(data, camera=close_camera)
    view2 = renderer.render().copy()
    merged = Image.new("RGB", (2 * W, H + 64), (16, 27, 41))
    merged.paste(Image.fromarray(view1), (0, 64))
    merged.paste(Image.fromarray(view2), (W, 64))
    draw = ImageDraw.Draw(merged)
    draw.text((12, 5), f"UR5e + Robotiq 2F-85 | {phase}", font=font, fill="white")
    draw.text((12, 35), "全体 (left)", font=small_font, fill=(177, 216, 235))
    draw.text((W + 12, 35), f"指先拡大 (right) | ctrl={cmd}/255", font=small_font, fill=(177, 216, 235))
    return np.asarray(merged)

preview = render_pair()
display(Image.fromarray(preview))
print("装着・表示の確認用画像です。グリッパーの向き、先端との接合を確認してください。")

## セル5：AI推論 → UR5e移動 → 2F-85閉開 → HOME の連続テスト

- **移動指示**：「左」「右」についてそれぞれアームを目標関節角へ移動し、物体のない状態で閉→開しHOMEへ戻る。
- **あいまいな指示**：`REJECT` のとき、現在の目標を保持して新しい目標姿勢・グリッパー指令を出さない。接触や慣性により微小な物理運動が継続する可能性はある。
- **HOME指示**：アームを待機姿勢へ移動し、既存のグリッパー指令を維持。

**重要**：動画の「CLOSE」は実物体を把持したことを意味しない。移動軌道は関節角の単純な補間であり、衝突回避ではない。

In [ ]:
# セル5：実演する文を明示的に定義し、推論結果を事前確認する
DEMO_INSTRUCTIONS = [
    ("左側の部品を検査台の前へ動かして", "LEFT_SAMPLE"),
    ("右にある試料を計測場所へ移動して", "RIGHT_SAMPLE"),
    ("右と左のどちらかの試料を運んで", "REJECT"),
    ("最初の姿勢へ戻って", "HOME"),
]

DEMO_INFERENCES = []
for instruction, expected in DEMO_INSTRUCTIONS:
    result = infer_task(instruction)
    print(f"指示：{instruction} / AI判定={result['task_id']} / 期待={expected} / 類似度={result['score']} / margin={result['margin']}")
    if result["task_id"] != expected:
        raise RuntimeError(
            f"実演指示へのAI応答が期待と不一致：{instruction!r}, 期待={expected}, 実際={result['task_id']}。"
            " 制御動画の生成を開始せず、ここで停止してください。"
        )
    DEMO_INFERENCES.append(result)
print("事前チェック成功。動画生成に進めます。これは選定済み例文のチェックであり独立評価ではありません。")

In [ ]:
# セル5B：関節目標とレンダリング・ログ関数
import os
import imageio.v2 as imageio
from PIL import Image, ImageDraw
from IPython.display import Video, display

FPS = 8
OUTPUT_VIDEO = "ur5e_ai_2f85_integrated_test.mp4"
OUTPUT_FRAMES = "ur5e_ai_2f85_integrated_frames.csv"
OUTPUT_TASKS = "ur5e_ai_2f85_integrated_tasks.csv"

# 02と同様の目標関節角。03のHOME_Q（6要素）から一部を変更する。
ARM_TARGETS = {"HOME": HOME_Q.copy()}
ARM_TARGETS["LEFT_SAMPLE"] = HOME_Q.copy()
ARM_TARGETS["LEFT_SAMPLE"][0] = -2.25
ARM_TARGETS["LEFT_SAMPLE"][1] = -1.30
ARM_TARGETS["RIGHT_SAMPLE"] = HOME_Q.copy()
ARM_TARGETS["RIGHT_SAMPLE"][0] = -0.90
ARM_TARGETS["RIGHT_SAMPLE"][1] = -1.30

for name, target in ARM_TARGETS.items():
    for aid, angle in zip(arm_actuator_ids, target):
        low, high = model.actuator_ctrlrange[aid]
        if not (low <= angle <= high):
            raise ValueError(f"目標角が制御範囲外: {name}: angle={angle}, range={low, high}")

assert model.actuator_ctrlrange[finger_actuator_id, 0] <= 0 <= model.actuator_ctrlrange[finger_actuator_id, 1]
assert model.actuator_ctrlrange[finger_actuator_id, 0] <= 255 <= model.actuator_ctrlrange[finger_actuator_id, 1]

# 各指示が選ばれたタスクに基づき異なる制御系列を選ぶ。
def get_action_plan(selected_task):
    if selected_task == "REJECT":
        return [("REJECT_HOLD", 2.0, None, None)]
    if selected_task == "HOME":
        return [("HOME", 2.0, ARM_TARGETS["HOME"], None)]
    if selected_task in {"LEFT_SAMPLE", "RIGHT_SAMPLE"}:
        return [
            ("ARM_MOVE", 2.5, ARM_TARGETS[selected_task], 0.0),
            ("GRIP_CLOSE", 1.6, ARM_TARGETS[selected_task], 255.0),
            ("GRIP_OPEN", 1.6, ARM_TARGETS[selected_task], 0.0),
            ("RETURN_HOME", 2.5, ARM_TARGETS["HOME"], 0.0),
        ]
    raise ValueError(f"未定義のタスクID: {selected_task}")

# 03と同じ2画面に、AIの入力と判定を追加する。
def render_integrated(instruction, selected, phase, ctrl):
    base = Image.fromarray(render_pair(phase, int(round(ctrl)))).convert("RGB")
    frame = Image.new("RGB", (2 * W, H + 64 + 102), (16, 27, 41))
    frame.paste(base, (0, 0))
    draw = ImageDraw.Draw(frame)
    draw.text((12, H+72), "AI指示: " + instruction, font=font, fill=(235, 247, 255))
    draw.text((12, H+102), f"AI判定={selected}   実行段階={phase}   grip ctrl={int(round(ctrl))}/255", font=small_font, fill=(160, 224, 244))
    draw.text((12, H+128), "AIはタスク選択のみ / 関節目標は固定 / 把持物体なし / 実機制御なし", font=small_font, fill=(250, 219, 156))
    return np.asarray(frame)

print("関節目標:")
for name, q in ARM_TARGETS.items():
    print(name, np.round(q, 3))
print("動画出力:", OUTPUT_VIDEO)

In [ ]:
# セル5C：MP4・フレーム別CSV・指示別CSVを生成
# 事前チェックをクリアしたときだけこのセルを実行する。
from IPython.display import Video, display

# 初期条件を毎回再現できるようリセットする。
mujoco.mj_resetData(model, data)
for qid, target in zip(arm_qpos_ids, HOME_Q):
    data.qpos[qid] = target
for aid, target in zip(arm_actuator_ids, HOME_Q):
    data.ctrl[aid] = target
data.ctrl[finger_actuator_id] = 0.0
mujoco.mj_forward(model, data)

rows = []
task_rows = []
frame_idx = 0
h = float(model.opt.timestep)
if h <= 0:
    raise ValueError("不正なシミュレーション時間刻み")

# 物理時刻を動画フレームの時刻へ合わせる。現場の実時間制御を模倣するものではない。
def execute_phase(writer, *, instruction, selected, phase, seconds, target_arm, target_grip, result):
    global frame_idx
    start_arm_ctrl = np.array([data.ctrl[a] for a in arm_actuator_ids], dtype=float)
    start_grip_ctrl = float(data.ctrl[finger_actuator_id])
    planned_arm = start_arm_ctrl.copy() if target_arm is None else np.asarray(target_arm, dtype=float)
    planned_grip = start_grip_ctrl if target_grip is None else float(target_grip)
    nframes = int(round(seconds * FPS))
    for i in range(nframes):
        frac = (i+1) / nframes
        smooth = frac*frac*(3-2*frac)
        for aid, angle in zip(arm_actuator_ids, start_arm_ctrl + (planned_arm-start_arm_ctrl)*smooth):
            data.ctrl[aid] = angle
        data.ctrl[finger_actuator_id] = planned_grip
        desired = (frame_idx+1)/FPS
        while data.time + 0.5*h < desired:
            mujoco.mj_step(model, data)
        if not np.all(np.isfinite(data.qpos)):
            raise RuntimeError(f"{phase}: 関節データに非有限値を検出。動画生成を停止")
        writer.append_data(render_integrated(instruction, selected, phase, planned_grip))
        actual_arm = np.asarray(data.qpos[arm_qpos_ids], dtype=float)
        rows.append({
            "time_s": round(float(data.time), 4),
            "instruction": instruction,
            "task_id": selected,
            "phase": phase,
            "score": result["score"],
            "margin": result["margin"],
            "gripper_ctrl_0_255": planned_grip,
            "left_driver_rad": float(data.qpos[left_qpos_id]),
            "right_driver_rad": float(data.qpos[right_qpos_id]),
            "max_arm_goal_error_rad": float(np.max(np.abs(actual_arm - planned_arm))),
            "max_arm_ctrl_tracking_error_rad": float(np.max(np.abs(actual_arm - [data.ctrl[a] for a in arm_actuator_ids]))),
            "contacts_n": int(data.ncon),
        })
        frame_idx += 1
    return float(np.max(np.abs(np.asarray(data.qpos[arm_qpos_ids])-planned_arm)))

with imageio.get_writer(OUTPUT_VIDEO, fps=FPS, codec="libx264", quality=7, macro_block_size=1) as writer:
    intro = {"score": 0.0, "margin": 0.0}
    execute_phase(writer, instruction="第4回実習：AI＋UR5e＋2F-85", selected="INTRO", phase="READY", seconds=1.0,
                  target_arm=HOME_Q, target_grip=0.0, result=intro)
    for (instruction, expected), result in zip(DEMO_INSTRUCTIONS, DEMO_INFERENCES):
        selected = result["task_id"]
        assert selected == expected
        for phase, seconds, target_arm, target_grip in get_action_plan(selected):
            ending_error = execute_phase(
                writer, instruction=instruction, selected=selected, phase=phase,
                seconds=seconds, target_arm=target_arm, target_grip=target_grip, result=result
            )
            task_rows.append({
                "instruction": instruction,
                "task_id": selected,
                "phase": phase,
                "score": result["score"],
                "margin": result["margin"],
                "end_arm_goal_error_rad": round(ending_error, 4),
                "end_left_driver_rad": round(float(data.qpos[left_qpos_id]), 4),
                "end_right_driver_rad": round(float(data.qpos[right_qpos_id]), 4),
                "end_gripper_cmd": round(float(data.ctrl[finger_actuator_id]), 1),
            })
    execute_phase(writer, instruction="連続動作テスト終了", selected="OUTRO", phase="END", seconds=1.0,
                  target_arm=HOME_Q, target_grip=0.0, result=intro)

frame_df = pd.DataFrame(rows)
task_df = pd.DataFrame(task_rows)
frame_df.to_csv(OUTPUT_FRAMES, index=False, encoding="utf-8-sig")
task_df.to_csv(OUTPUT_TASKS, index=False, encoding="utf-8-sig")

print("動画:", OUTPUT_VIDEO, f"{frame_idx/FPS:.2f}秒 / {frame_idx}フレーム")
print("フレームログ:", OUTPUT_FRAMES, len(frame_df), "行")
print("段階別ログ:", OUTPUT_TASKS, len(task_df), "行")
print(task_df.to_string(index=False))

# 動作の自動判定は目標関節到達の簡易的な目安で、安全性や把持成功を保証しない。
last_moves = task_df[task_df.phase.isin(["ARM_MOVE", "RETURN_HOME", "HOME"])]
if not last_moves.empty:
    worst = float(last_moves.end_arm_goal_error_rad.max())
    print(f"移動フェーズ終了時の最大姿勢誤差 = {worst:.4f} rad")
    print("簡易目標到達 (<0.12 rad):", bool(worst < 0.12))
print("注：REJECTには新しい関節目標や開閉指令を発行していません。")
display(Video(OUTPUT_VIDEO, embed=True, width=960))

## セル6：グラフとCSVログを確認
閉開が実際に追従しているか、グリッパー入力0↔255と左右関節角の変化を分けて見る。`contacts_n` はロボット自身の接触も含み、**把持成功回数ではない**。

In [ ]:
# セル6：入力と関節角をグラフ化
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].step(frame_df.time_s, frame_df.gripper_ctrl_0_255, where="post", label="2F-85 command [0..255]")
axes[0].set_ylabel("command")
axes[0].legend(loc="upper right")
axes[0].grid(alpha=0.3)
axes[1].plot(frame_df.time_s, frame_df.left_driver_rad, label="left finger [rad]")
axes[1].plot(frame_df.time_s, frame_df.right_driver_rad, label="right finger [rad]", linestyle="--")
axes[1].set_xlabel("Simulation time [s]")
axes[1].set_ylabel("driver joint [rad]")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("左右指の全変化量[rad]:", round(float(frame_df.left_driver_rad.max()-frame_df.left_driver_rad.min()),4), round(float(frame_df.right_driver_rad.max()-frame_df.right_driver_rad.min()),4))
print("REJECTフレーム数:", int((frame_df.phase == "REJECT_HOLD").sum()))

## セル7：Google ColabからPCに保存
まず動画とCSVをダウンロードする。GitHubにはこの04ノートブックと結果CSVを**動作検証後**に登録する。既存の03のログや02の提出用動画は上書きしない。

In [ ]:
# セル7：PCへ動画・CSVをダウンロード（Colab専用）
from google.colab import files
files.download(OUTPUT_VIDEO)
files.download(OUTPUT_FRAMES)
files.download(OUTPUT_TASKS)

## 成功判定・研究上の限界

**成功と判断できる条件（いずれも確認が必要）**

1. AIが4つの事前選定文を `LEFT_SAMPLE`、`RIGHT_SAMPLE`、`REJECT`、`HOME` と判断する。
2. UR5e先端に2F-85が正しく装着されている。
3. 左右の各タスクで姿勢移動の後にグリッパーが閉じ、開き、アームがHOMEへ戻る。
4. REJECTの際には新たな目標姿勢・開閉を指令せず保持する。
5. MP4とCSVが生成され、ログの関節角の推移が動画に対応する。

**評価上の留意点**

- 選定済みの4文に対する成功は独立した推論精度の評価にはならない。第2回の未知10文評価は6/10であり、分類の改善を要する。
- 目標関節角はプログラムに手動設定した値であり、試料位置を検出していない。移動指示の「運ぶ」は文字通りの搬送ではない。
- 指を閉じても把持物体はないため、物体の把持・搬送性能は示せない。
- ロボット実機での可動域制限、接触力、速度・加速度制約、特異点、安全対策を検証していない。
- 次段階の実験は、衝突判定・力制御・幾何学的な試料配置・IK・簡易Pick & Placeを別途設計する。

**出典／元モデル**：Google DeepMind MuJoCo Menagerie の`universal_robots_ur5e`と`robotiq_2f85`、Sentence Transformers の`paraphrase-multilingual-MiniLM-L12-v2`。実際の依存バージョンはセル1を参照。